### Connection MYSQL

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy import text

In [ ]:
load_dotenv()

DB_USER = os.getenv("DB_USER");
DB_PASSWORD = os.getenv("DB_PASSWORD");
DB_HOST = os.getenv("DB_HOST");
DB_PORT = os.getenv("DB_PORT", 3306);
DB_SIAKAD = os.getenv("DB_SIAKAD");

siakadConnection = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_SIAKAD}"
siakadEngine = create_engine(siakadConnection)

### Library

In [ ]:
import pandas as pd
import json

### Dosen Tetap Program Studi (DTPS)

In [ ]:

def parse_details(x):
  if pd.isna(x):
    return {}
  try:
    data = json.loads(x)
    for key in ['serdos', 'jabfung', 'gelar_depan', 'gelar_belakang']:
      data.pop(key, None)
    return data
  except Exception:
    return {}

In [ ]:
jenjangPendidikanQuery = text("""
SELECT code, name FROM jenjang_pendidikan
""")

jenjangPendidikanDf = pd.read_sql(jenjangPendidikanQuery, siakadConnection)

In [ ]:
dtpsQuery = text("""
SELECT
    dosen.name AS nama_dosen,
    homebase.name AS homebase,
    semesters.year_of_study AS tahun_ajaran,
    kelas_prodi.name AS kelas_prodi,
    matkul.name AS matakuliah,
    matkul.jenis_matakuliah_code,
    jenis_matkul.name AS jenis_matkul,
    dosen.details
FROM dosen_list AS dosen

JOIN prodi AS homebase ON
    dosen.prodi_uuid = homebase.uuid

JOIN dosen_kelas_laporan AS dsl ON
    dsl.dosen_uuid = dosen.uuid

JOIN kelas_laporan ON
    dsl.kelas_uuid = kelas_laporan.uuid

JOIN matakuliah AS matkul ON
    kelas_laporan.matkul_uuid = matkul.uuid

JOIN semesters ON
    kelas_laporan.semester_code = semesters.code

JOIN prodi AS kelas_prodi ON
    kelas_laporan.prodi_uuid = kelas_prodi.uuid

LEFT JOIN matakuliah_jenis AS jenis_matkul ON
    matkul.jenis_matakuliah_code = jenis_matkul.code

WHERE
    dosen.active = 1 AND
    semesters.year_of_study >= ( YEAR ( CURDATE()) - 4 ) AND
    dsl.deleted_at IS NULL

ORDER BY semesters.year_of_study DESC
""")

dtpsDf = pd.read_sql(dtpsQuery, siakadEngine);
details = dtpsDf['details'].apply(parse_details)
details = pd.json_normalize(details)

dtpsDf = pd.concat(
    [dtpsDf.drop(columns=['details']), details], axis=1
)

dtpsDf['pendidikan_terakhir'] = dtpsDf['pendidikan_terakhir'].astype(str)
jenjangPendidikanDf['code'] = jenjangPendidikanDf['code'].astype(str)

dtpsDf = dtpsDf.merge(
    jenjangPendidikanDf, 
    left_on='pendidikan_terakhir', 
    right_on='code', 
    how='left'
)

dtpsDf = dtpsDf.drop(columns=['code'])
dtpsDf = dtpsDf.drop(columns=['pendidikan_terakhir'])
dtpsDf = dtpsDf.rename(columns={'name': 'pendidikan_terakhir'});

In [ ]:
dtpsDf.to_csv('data/dosen/dosen-matakuliah.csv',index=False)

### Rasio Dosen Terhadap Mahasiswa

In [ ]:
rasioDosenQuery = text("""
SELECT
    semesters.year_of_study AS tahun_sempurna,
    prodi.name AS prodi,
    dsl.dosen_uuid,
    nilai.mahasiswa_register_id,
    dosen.details

FROM kelas_laporan

JOIN semesters ON
    kelas_laporan.semester_code = semesters.code

JOIN kelas ON
    kelas_laporan.uuid = kelas.kelas_laporan_uuid

JOIN mahasiswa_nilais AS nilai ON
    kelas.id = nilai.kelas_id

JOIN dosen_kelas_laporan AS dsl ON
    dsl.kelas_uuid = kelas_laporan.uuid

JOIN dosen_list AS dosen ON
    dsl.dosen_uuid = dosen.uuid

JOIN prodi ON
    dosen.prodi_uuid = prodi.uuid

WHERE
    semesters.year_of_study >= ( YEAR ( CURDATE()) - 4 ) AND
    kelas_laporan.deleted_at IS NULL AND
    kelas.deleted_at IS NULL AND
    nilai.deleted_at IS NULL AND
    dsl.deleted_at IS NULL AND
    dosen.deleted_at IS NULL
""")

rasioDosenDf = pd.read_sql(rasioDosenQuery, siakadEngine)
details = rasioDosenDf['details'].apply(parse_details)
details = pd.json_normalize(details)

rasioDosenDf = pd.concat(
    [rasioDosenDf.drop(columns=['details']), details], axis=1
)

rasioDosenDf = rasioDosenDf[rasioDosenDf['status_pegawai'] == 'dosen_tetap']

In [ ]:
rasioDosenDf.drop(columns=['pendidikan_terakhir', 'status_pegawai']).to_csv('data/dosen/rasio-dosen.csv', index=False)